# Harness Bench

**A very small benchmark to find the harness you need.**

The saved campaign compares four harnesses with the model alone (`bare`): one call, no tools. It shows what a harness adds.

Choose the project's `.venv` Python kernel. This notebook opens saved results by default. It never launches a campaign until you set `run_new_campaign = True`. Each work score retains errors, timeouts and unexecuted trials in its denominator. The startup check is reported separately.


In [ ]:
from pathlib import Path
import importlib
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/local_comparison.py").is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "scripts"))
import bench
import local_comparison
import scripts.campaign_results as campaign_results
importlib.reload(campaign_results)
importlib.reload(bench)
importlib.reload(local_comparison)
print("Python kernel", sys.executable)

## Choose the experiment

`demo` contains two tasks, `smoke` three, `quick` eight and `full` twenty-one (three per category: startup, coding, data science, research, general work, context retrieval and math, each with at least one easy task). `math` is a three-task suite that shows whether a harness gets the model to run code. `value` has six tasks, three that need running code and three that do not, and is meant to be run with `bare`, the model alone, as the baseline. Quick covers every category except a second data-science task, plus a separate startup check. The preview below shows the exact task instructions and campaign size before anything runs.

Leave `run_new_campaign = False` to inspect saved results. Set `campaign_id` to a saved timestamp, or leave it as `None` for the newest local campaign. A new campaign records requested limits and available Ollama runtime metadata. Actual context observations can be absent when the server unloads a model.

### What `value` runs

`value` = **6 tasks × 5 configurations × 3 attempts = 90 trials**, roughly 2 to 4 hours with a local model, because `bare` spends up to two minutes on a task it cannot solve.

### What `full` runs

`full` = **21 tasks × 4 harnesses × 3 attempts = 252 trials**. The startup tasks run first as a health check; the others follow in a fixed shuffled order.

- **Local model:** trials run one at a time, so every harness meets the same hardware. The agent CLIs are preinstalled in the shared Docker image (built once, a few minutes), so setup takes about a second per trial. Expect **roughly 5 hours**. A trial that hits the 600 s limit counts as a timeout, so the worst case is longer. Keep the computer awake and plugged in.
- **Ollama cloud model** (`model = "glm-5.3-flash:cloud"`): run `ollama signin` once and add credits at ollama.com/plans. Up to **3 trials run at once**, the concurrent-request limit of the pay-as-you-go and Pro plans (Max allows 10; set `concurrency`). Each worker has its own measuring proxy. Calls are billed per token, so check the price of your model first. Latency and tokens per second then describe the network and the service, not your machine.
- **Cost:** none locally, only energy and Docker disk. A cloud campaign is billed by Ollama.
- **To launch:** set `run_new_campaign = True` and run the cells from top to bottom. The progress bar appears under the "Run or select a campaign" cell.
- **Smaller first:** change `suite` to `math` (3 tasks) or `quick` (8 tasks) to check the setup in minutes.


In [ ]:
model = "lfm3:1.7b"  # local, or an Ollama cloud model such as "glm-5.3-flash:cloud" (needs `ollama signin` and credits)
harnesses = ["bare", "pi", "opencode", "droid", "dsh"]  # bare = the model alone, no tools
suite = "value"  # demo 2 tasks, smoke 3, math 3, value 6, quick 8, full 21
attempts = 3  # repeat each task so noise does not decide the ranking
timeout_sec = 600
provider = None  # None = local Ollama (or an Ollama cloud tag); "gemini" calls the Gemini API with GEMINI_API_KEY from .env
concurrency = None  # trials at once; None = 1 for a local model, 3 for an Ollama cloud model
context_window = 32768
max_output_tokens = 8192
diagnostic_task = "math-count"  # Choose one task name, or auto-select a work task.

run_new_campaign = False  # Change to True when you want to launch.
campaign_id = "20261123T095726180399Z"  # the finished value campaign; None = latest saved
resume_campaign = None  # e.g. "20261118T102128134046Z" to continue an interrupted run

In [ ]:
from IPython.display import HTML, display
import html
import tomllib

plan = bench.build_plan(suite, harnesses, dict.fromkeys(harnesses, model), attempts, local=True)
settings, _ = bench.read_config()
task_names = settings["suites"][suite]
work_count = sum(name != "startup" for name in task_names)
print(f"Preview only: {plan['runs']} sequential trials, {work_count} work tasks per harness.")
print(f"Agent time limit total: {plan['runs'] * timeout_sec / 60:.0f} minutes if every trial times out.")
print("Setup and grading add time. Local inference has no provider token charge; hardware uses energy.")
for name in task_names:
    folder = root / "tasks" / name
    metadata = tomllib.loads((folder / "task.toml").read_text())["metadata"]
    instructions = (folder / "instruction.md").read_text()
    display(HTML("<details><summary><strong>" + html.escape(name) + "</strong> · "
                 + html.escape(metadata["category"]) + "</summary><pre style='white-space:pre-wrap'>"
                 + html.escape(instructions) + "</pre></details>"))


## Prepare the notebook

This cell installs matplotlib and ipywidgets (for the graphical progress bar) in the selected Python environment if needed. Docker and Ollama preparation happens only when you request a new campaign. Start their applications first. Before a new campaign, it also updates stale execution libraries from the lockfile. Existing reports do not require an execution-library update.

In [ ]:
import importlib.util
import json
import shutil
import subprocess
import urllib.request

installer = (["uv", "pip", "install", "--python", sys.executable]
             if shutil.which("uv") else [sys.executable, "-m", "pip", "install"])
def install_notebook_packages(arguments):
    if installer[0] == sys.executable and importlib.util.find_spec("pip") is None:
        subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
    subprocess.run(installer + arguments, check=True)

if importlib.util.find_spec("matplotlib") is None:
    install_notebook_packages(["matplotlib>=3.9,<4"])
if importlib.util.find_spec("ipywidgets") is None:
    install_notebook_packages(["ipywidgets>=8,<9"])  # graphical tqdm bar; restart the kernel once if it still looks like text

if run_new_campaign:
    try:
        bench.runtime_versions()
    except ValueError:
        print("Updating execution dependencies before Docker and model setup.")
        install_notebook_packages(["-r", str(root / "requirements.lock")])
        bench.runtime_versions()
    subprocess.run(["docker", "info"], check=True, capture_output=True)
    # The shared task image with the agent CLIs preinstalled is built, or refreshed, when the campaign starts.
    if provider is None and not model.endswith((":cloud", "-cloud")):
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as response:
            installed_models = json.load(response)["models"]
        if model not in {item["name"] for item in installed_models}:
            subprocess.run(["ollama", "pull", model], check=True)


## Run or select a campaign

**The progress bar** (tqdm) counts finished trials out of the total, with elapsed time and an estimate of the time left. Next to it you see the trial running now (harness / task), the latest native event and how long the log has been silent, the last finished trial with its outcome, and a running tally such as `12 passed, 3 incorrect, 1 timeout`. The estimate gets better after a few trials, because harness and task durations vary a lot.

**Nothing is lost if it stops.** Each trial is saved the moment it ends, and `results.csv` and `report.md` in the campaign folder are refreshed after every trial. If the kernel stops, the computer sleeps or you interrupt the cell, set `run_new_campaign = True` and `resume_campaign = "<campaign folder name>"`, then rerun. Finished trials are kept; only interrupted or technically failed ones are redone. The campaign keeps its original model, settings and harness versions, and refuses to resume if a task or the model weights changed. A harness with three technical errors in a row is skipped for the rest of the run instead of wasting hours.

Interrupt the kernel to stop. Open saved results at any time by leaving `run_new_campaign = False`.


In [ ]:
if run_new_campaign:
    campaign = local_comparison.run_comparison(
        model=model, harnesses=harnesses, suite=suite, attempts=attempts,
        timeout_sec=timeout_sec, context_window=context_window,
        max_output_tokens=max_output_tokens,
        resume=(root / "runs/campaigns" / resume_campaign) if resume_campaign else None,
        concurrency=concurrency, provider=provider,
    )
else:
    available = local_comparison.saved_campaigns()
    campaign = (root / "runs/campaigns" / campaign_id if campaign_id
                else next(iter(available), None))
if campaign is None:
    raise ValueError("No local campaign saved yet. Set run_new_campaign = True to start.")
print("Campaign", campaign)
rows = campaign_results.load_rows(campaign)

## Read the evidence

`bare` is not a harness: it is the model alone, with one call and no tools, so its row shows what the model does without help. The gap between it and a harness is what that harness adds.

The table and overview score **work tasks only**. Startup is a health check shown separately. A success requires every required check to pass. Timeouts, technical errors and unfinished scheduled work count against that work score. Unknown measurements stay unknown, with observed counts shown alongside cost and runtime.

The thin lines are exploratory 95% Wilson intervals for this set of trials. Small samples produce wide intervals even at 0% or 100%. Task attempts can be dependent, so these intervals do not establish a general harness ranking. Compare the same model, task mix and limits, then inspect the outputs.

Category and task heatmaps show each denominator explicitly. Gray N/A means no applicable trial or unavailable diagnostic. The diagnostics panel covers one task selected with `diagnostic_task`; the panel title identifies it. Diagnostic percentages use only recorded checks and are not partial-credit grades. Older retrieval campaigns can recover limited public diagnostics from saved answers; their hidden checks remain unavailable.

The efficiency figure uses a local logging proxy placed between every harness and Ollama, so all harnesses are measured the same way. *LLM calls* are model requests, *tool calls* are the tool invocations the model requested, *time to first call* runs from the start of the agent phase to the first model request (it includes harness start-up), and *TTFT* is the delay before the first byte of each answer. Unknown stays unknown.


In [ ]:
from IPython.display import HTML, display
import matplotlib.pyplot as plt

print("Campaign:", campaign.name)
display(HTML(campaign_results.summary_html(rows)))
figure = campaign_results.plot_results(rows, diagnostic_task=diagnostic_task)
display(figure)
plt.close(figure)

**How to read the overview figure.**
- *Work success* (top left): the share of work tasks each harness passed. The black line is a 95% interval; with so few tasks it is wide, so small gaps mean little.
- *Work outcomes* (top right): the same trials split into passed, incorrect, timeout, technical error and not run. This shows *how* a harness fails, not only how often.
- *Median work time* (bottom left): how long the agent typically needs per task. Shorter is faster, not necessarily better.
- *Diagnostics* (bottom right): for one task, which individual checks passed. Green means the check passed in most trials, red means it failed.

In [ ]:
task_figure = campaign_results.plot_task_results(rows)
display(task_figure)
plt.close(task_figure)

**How to read the task figure.** Each row is a harness and each cell is the share of trials it passed, with passed/scheduled in the cell. Dark blue is high, pale is low, grey N/A means nothing was run. The left map groups tasks by category and the right map shows single tasks. Look for columns that are pale for every harness (hard tasks) and for rows that differ from the others on the same task (a harness-specific strength or weakness).

In [ ]:
efficiency_figure = campaign_results.plot_efficiency(rows)
display(efficiency_figure)
plt.close(efficiency_figure)

# Optional exports. Uncomment to save locally; do not publish private trial outputs.
# figure.savefig(campaign / "overview.png", dpi=160, bbox_inches="tight")
# task_figure.savefig(campaign / "tasks.png", dpi=160, bbox_inches="tight")


**How to read the efficiency figure.** These numbers come from a small logging proxy placed in front of the model, so every harness is measured the same way.
- *LLM calls* and *tool calls*: how many times the harness asks the model and how many actions the model requests. More calls usually means a longer loop.
- *Time to first call*: how long the harness takes to start before its first model request.
- *Time to first token*: how long the model takes to begin answering one call.
- *Call latency*: how long one whole model call takes.
- *Tokens per second*: how fast the model writes while it is answering.
A harness can be cheap per call but slow overall if it makes many calls.

## Matched comparisons

Each row compares two harnesses on the same task and attempt. *left wins* counts trials only the left harness passed, *right wins* the reverse, and ties are trials with the same outcome. *p value* is an exact sign test on the wins: below 0.05 suggests a real difference, above it the gap may be chance. Trials that did not run are excluded. Rows with `bare` show what a harness adds over the model alone.

In [ ]:
pairs = campaign_results.paired_outcomes(rows)
if pairs:
    columns = ["left", "right", "model", "matched", "left_wins", "right_wins", "ties", "p_value", "unmatched"]
    header = "".join("<th>" + html.escape(key.replace("_", " ")) + "</th>" for key in columns)
    body = "".join("<tr>" + "".join("<td>" + html.escape("" if pair[key] is None else (f"{pair[key]:.2f}" if key == "p_value" else str(pair[key]))) + "</td>"
                                  for key in columns) + "</tr>" for pair in pairs)
    display(HTML('<table class="hb-table"><thead><tr>' + header + "</tr></thead><tbody>"
                 + body + "</tbody></table>"))
else:
    print("No configurations with the same model tag to pair.")


## Inspect each trial

Use the logs folder to inspect native events, exception.txt, verifier output and generated programs. A timeout may still have partial files. Review the error and diagnostics before comparing harnesses.

In [ ]:
import html
import os
from urllib.parse import quote
from IPython.display import HTML

columns = ["harness", "task", "attempt", "status", "reward", "agent_seconds",
           "llm_calls", "tool_calls", "first_call_seconds", "input_tokens", "output_tokens"]
header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
body = []
for row in rows:
    values = "".join("<td>" + html.escape(str("unknown" if row.get(key) in (None, "") else row[key])) + "</td>"
                     for key in columns)
    result = Path(row["result"]) if row.get("result") else None
    if result and result.exists():
        url = os.path.relpath(result.parent, Path.cwd())
        link = '<a href="' + html.escape(quote(url), quote=True) + '">Open trial files</a>'
    else:
        link = "No trial files"
    body.append("<tr>" + values + "<td>" + link + "</td></tr>")
display(HTML("<table><thead><tr>" + header + "<th>Logs</th></tr></thead><tbody>"
             + "".join(body) + "</tbody></table>"))
for row in rows:
    checks = campaign_results.diagnostic_checks(row)
    print(row["harness"], row["task"], checks)
    if row.get("error_message"):
        print(row["error_message"])